# Weather Competition Jupyter Runner

Run these cells from top to bottom after uploading the project to JupyterLab. Start with the smoke test settings, then switch to full training after the environment and data paths are confirmed.

In [3]:
# 1. Check runtime
import os
import sys
import torch

print(sys.version)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
torch: 2.12.1+cu132
cuda: True
gpu: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [ ]:
# 2. Check project and platform paths
from pathlib import Path

ROOT = Path('.').resolve()
print('ROOT:', ROOT)
print([p.name for p in ROOT.iterdir()])

!python scripts/check_platform_paths.py

In [ ]:
# 3. Set paths if the platform dataset is not under data/train and data/test
# os.environ['TRAIN_DIR'] = 'actual_train_dir'
# os.environ['TEST_DIR'] = 'actual_test_dir'
# os.environ['SAMPLE_SUBMISSION_PATH'] = 'actual_sample_submission.csv'

print('TRAIN_DIR =', os.environ.get('TRAIN_DIR', 'data/train'))
print('TEST_DIR =', os.environ.get('TEST_DIR', 'data/test'))
print('SAMPLE_SUBMISSION_PATH =', os.environ.get('SAMPLE_SUBMISSION_PATH', 'data/sample_submission.csv'))

In [ ]:
# 4. Check training data
from src.dataset import scan_image_folder

train_dir = os.environ.get('TRAIN_DIR', 'data/train')
df = scan_image_folder(train_dir)
print(df['label'].value_counts().sort_index())
print('total:', len(df))

In [ ]:
# 5. Lightweight smoke-test parameters for first JupyterLab run
os.environ['MODEL_NAME'] = 'efficientnet_b0'
os.environ['FALLBACK_MODEL_NAME'] = 'efficientnet_b0'
os.environ['PRETRAINED'] = 'false'
os.environ['IMG_SIZE'] = '224'
os.environ['BATCH_SIZE'] = '8'
os.environ['EPOCHS'] = '1'
os.environ['NUM_WORKERS'] = '0'

for key in ['MODEL_NAME', 'PRETRAINED', 'IMG_SIZE', 'BATCH_SIZE', 'EPOCHS', 'NUM_WORKERS']:
    print(key, os.environ[key])

In [ ]:
# 6. Run isolated synthetic smoke test
# This does not need official test images and writes generated files under tmp/smoke_test.
!python scripts/smoke_test.py --epochs 1 --batch-size 8 --model-name efficientnet_b0 --fallback-model-name efficientnet_b0

In [4]:
# 7. Switch to full training settings after smoke test passes
# EfficientNetV2-S full-training settings. Adjust BATCH_SIZE if GPU memory is not enough.
if not torch.cuda.is_available():
    raise RuntimeError('Current Jupyter kernel cannot see CUDA. Restart kernel or use a GPU runtime.')

os.environ['DEVICE'] = 'cuda'
os.environ['REQUIRE_CUDA'] = 'true'
os.environ['MODEL_NAME'] = 'tf_efficientnetv2_s'
os.environ['FALLBACK_MODEL_NAME'] = 'efficientnet_b0'
os.environ['PRETRAINED'] = 'true'
os.environ['IMG_SIZE'] = '224'
os.environ['BATCH_SIZE'] = '8'
os.environ['EPOCHS'] = '30'
os.environ['NUM_WORKERS'] = '2'
os.environ['USE_AMP'] = 'true'
os.environ['USE_CLASS_WEIGHT'] = 'true'

for key in ['DEVICE', 'REQUIRE_CUDA', 'MODEL_NAME', 'FALLBACK_MODEL_NAME', 'PRETRAINED', 'IMG_SIZE', 'BATCH_SIZE', 'EPOCHS', 'NUM_WORKERS', 'USE_AMP', 'USE_CLASS_WEIGHT']:
    print(key, os.environ[key])

DEVICE cuda
REQUIRE_CUDA true
MODEL_NAME tf_efficientnetv2_s
FALLBACK_MODEL_NAME efficientnet_b0
PRETRAINED true
IMG_SIZE 224
BATCH_SIZE 8
EPOCHS 30
NUM_WORKERS 2
USE_AMP true
USE_CLASS_WEIGHT true


In [5]:
# 8. Run full training in the current Jupyter kernel for live progress
# This avoids subprocess output buffering and shows tqdm progress in real time.
import importlib
import config
import train

importlib.reload(config)
importlib.reload(train)
train.main()

device=cuda
gpu=NVIDIA GeForce RTX 5070 Ti Laptop GPU, total_memory=11.9GB


E:\code\weather\weather_competition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\code\weather\weather_competition\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aloha\.cache\huggingface\hub\models--timm--tf_efficientnetv2_s.in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrat

Model: tf_efficientnetv2_s, trainable parameters: 20,182,612
Class weights: [0.5723000168800354, 2.8004000186920166, 3.10479998588562, 0.6355999708175659]


Validate: 100%|██████████| 125/125 [00:07<00:00, 17.76it/s]


{'epoch': 1, 'train_loss': 1.8976195903085535, 'train_accuracy': 0.6079019754938735, 'train_macro_f1': 0.5499508058009077, 'train_weighted_f1': 0.6319553395288123, 'val_loss': 0.9437953872680664, 'val_accuracy': 0.856, 'val_macro_f1': 0.8181246789261443, 'val_weighted_f1': 0.8590606248776009, 'lr': 0.0002991810233575568}


...

{'epoch': 24, 'train_loss': 0.616201515926901, 'train_accuracy': 0.989247311827957, 'train_macro_f1': 0.98849924594876, 'train_weighted_f1': 0.9892524327547352, 'val_loss': 0.7463098750114441, 'val_accuracy': 0.932, 'val_macro_f1': 0.9109972938659435, 'val_weighted_f1': 0.9325577145079599, 'lr': 2.9551959340945376e-05}


Validate: 100%|██████████| 125/125 [00:06<00:00, 19.62it/s]


{'epoch': 25, 'train_loss': 0.6082290817600812, 'train_accuracy': 0.9924981245311327, 'train_macro_f1': 0.9913699026515934, 'train_weighted_f1': 0.9924998074037253, 'val_loss': 0.7401131982803345, 'val_accuracy': 0.93, 'val_macro_f1': 0.9059773116751549, 'val_weighted_f1': 0.9303151343225897, 'lr': 2.102920213422642e-05}


Validate: 100%|██████████| 125/125 [00:06<00:00, 19.33it/s]


{'epoch': 26, 'train_loss': 0.606183898183637, 'train_accuracy': 0.9942485621405351, 'train_macro_f1': 0.9936057882500235, 'train_weighted_f1': 0.994251502857887, 'val_loss': 0.7379990603923797, 'val_accuracy': 0.932, 'val_macro_f1': 0.9080684105471648, 'val_weighted_f1': 0.9323275094224732, 'lr': 1.3924954082431156e-05}
              precision    recall  f1-score   support

      cloudy       0.95      0.93      0.94       437
       rainy       0.83      0.90      0.86        89
       snowy       0.85      0.93      0.89        81
       sunny       0.96      0.95      0.96       393

    accuracy                           0.93      1000
   macro avg       0.90      0.93      0.91      1000
weighted avg       0.94      0.93      0.93      1000



In [ ]:
# 9. Run inference after official test images are available
!python infer.py

In [ ]:
# 10. Download and prepare an external labeled public test set
# Source: Hugging Face davidshableski/weatherimages. The script keeps cloudy/rainy/snowy/sunny and skips sunrise.
!python scripts/download_public_weather_test.py --output-dir tmp/public_weather_test

In [ ]:
# 11. Evaluate the trained checkpoint on the external labeled test set
!python scripts/evaluate_labeled_folder.py --data-dir tmp/public_weather_test --output-dir outputs/external_test --batch-size 32 --num-workers 0

In [ ]:
# 12. Check generated files, submission CSV, and external test metrics
import pandas as pd

paths = [
    Path('results/best_model.pth'),
    Path('results/training_summary.json'),
    Path('logs/train_log.csv'),
    Path('outputs/submissions/submission.csv'),
    Path('outputs/external_test/external_test_metrics.json'),
    Path('outputs/confusion_matrix.png'),
]
for path in paths:
    print(path, path.exists())

sub_path = Path('outputs/submissions/submission.csv')
if sub_path.exists():
    sub = pd.read_csv(sub_path)
    print(sub.head())
    print(sub.shape)
    print(sub.isnull().sum())